In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

import joblib

In [3]:
price = pd.read_csv("../data/forex_price_data.csv")

price["DateTime"] = pd.to_datetime(price["DateTime"])

price.head()

,DateTime,PairID,PairName,Open,High,Low,Close,Volume,Spread
0,2023-01-01 00:00:00,1,EUR/USD,1.08103,1.08103,1.08019,1.08054,2651,0.9
1,2023-01-01 01:00:00,1,EUR/USD,1.08010,1.08043,1.07974,1.08028,2292,1.0
2,2023-01-01 02:00:00,1,EUR/USD,1.08036,1.08158,1.07961,1.08054,1796,1.2
3,2023-01-01 03:00:00,1,EUR/USD,1.08068,1.08126,1.07973,1.08088,4019,1.4
4,2023-01-01 04:00:00,1,EUR/USD,1.08042,1.08042,1.07967,1.08030,2207,1.6


In [4]:
# Moving Averages
price["MA_5"] = price["Close"].rolling(5).mean()
price["MA_20"] = price["Close"].rolling(20).mean()

# Feature Engineering
price["Price_Change"] = price["Close"] - price["Open"]
price["High_Low_Range"] = price["High"] - price["Low"]
price["Return"] = price["Close"].pct_change()

In [5]:
price["Target"] = np.where(
    price["Close"].shift(-1) > price["Close"],
    1,
    0
)

In [6]:
price = price.dropna().copy()

In [7]:
price[[
    "Close",
    "Target"
]].tail(10)

,Close,Target
490,1.11329,0
491,1.11173,0
492,1.10983,0
493,1.10950,1
494,1.10983,0
495,1.10861,0
496,1.10739,0
497,1.10713,1
498,1.10758,1
499,1.10895,0


In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from ta.momentum import RSIIndicator
from ta.trend import MACD
from ta.volatility import BollingerBands

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import joblib

In [9]:
price = pd.read_csv("../data/forex_price_data.csv")

price["DateTime"] = pd.to_datetime(price["DateTime"])

price.head()

,DateTime,PairID,PairName,Open,High,Low,Close,Volume,Spread
0,2023-01-01 00:00:00,1,EUR/USD,1.08103,1.08103,1.08019,1.08054,2651,0.9
1,2023-01-01 01:00:00,1,EUR/USD,1.08010,1.08043,1.07974,1.08028,2292,1.0
2,2023-01-01 02:00:00,1,EUR/USD,1.08036,1.08158,1.07961,1.08054,1796,1.2
3,2023-01-01 03:00:00,1,EUR/USD,1.08068,1.08126,1.07973,1.08088,4019,1.4
4,2023-01-01 04:00:00,1,EUR/USD,1.08042,1.08042,1.07967,1.08030,2207,1.6


In [10]:
# Moving Averages
price["MA_5"] = price["Close"].rolling(window=5).mean()
price["MA_20"] = price["Close"].rolling(window=20).mean()

# Feature Engineering
price["Price_Change"] = price["Close"] - price["Open"]
price["High_Low_Range"] = price["High"] - price["Low"]
price["Return"] = price["Close"].pct_change()

# RSI
price["RSI"] = RSIIndicator(
    close=price["Close"],
    window=14
).rsi()

# MACD
macd = MACD(close=price["Close"])

price["MACD"] = macd.macd()
price["MACD_Signal"] = macd.macd_signal()

# Bollinger Bands
bb = BollingerBands(
    close=price["Close"],
    window=20,
    window_dev=2
)

price["BB_Upper"] = bb.bollinger_hband()
price["BB_Middle"] = bb.bollinger_mavg()
price["BB_Lower"] = bb.bollinger_lband()

price.tail()

,DateTime,PairID,PairName,Open,High,Low,Close,Volume,Spread,MA_5,MA_20,Price_Change,High_Low_Range,Return,RSI,MACD,MACD_Signal,BB_Upper,BB_Middle,BB_Lower
495,2023-01-21 15:00:00,1,EUR/USD,1.10876,1.10912,1.10822,1.10861,2403,1.1,1.109900,1.113773,-0.00015,0.00090,-0.001099,28.186279,-0.001262,-0.000544,1.119466,1.113773,1.108081
496,2023-01-21 16:00:00,1,EUR/USD,1.10722,1.10751,1.10671,1.10739,2081,1.5,1.109032,1.113284,0.00017,0.00080,-0.001100,25.076850,-0.001505,-0.000737,1.119388,1.113284,1.107179
497,2023-01-21 17:00:00,1,EUR/USD,1.10730,1.10735,1.10687,1.10713,1828,1.1,1.108492,1.112755,-0.00017,0.00048,-0.000235,24.457616,-0.001700,-0.000929,1.119065,1.112755,1.106445
498,2023-01-21 18:00:00,1,EUR/USD,1.10729,1.10775,1.10712,1.10758,3894,1.7,1.108108,1.112228,0.00029,0.00063,0.000406,27.781563,-0.001796,-0.001103,1.118415,1.112228,1.106040
499,2023-01-21 19:00:00,1,EUR/USD,1.10871,1.10902,1.10829,1.10895,2615,1.4,1.107932,1.111777,0.00024,0.00073,0.001237,36.886526,-0.001742,-0.001231,1.117523,1.111777,1.106030


In [11]:
# Create Target Variable
price["Target"] = np.where(
    price["Close"].shift(-1) > price["Close"],
    1,
    0
)

# Remove rows with missing values
price = price.dropna().copy()

# Display last few rows
price[["DateTime", "Close", "Target"]].tail(10)

,DateTime,Close,Target
490,2023-01-21 10:00:00,1.11329,0
491,2023-01-21 11:00:00,1.11173,0
492,2023-01-21 12:00:00,1.10983,0
493,2023-01-21 13:00:00,1.10950,1
494,2023-01-21 14:00:00,1.10983,0
495,2023-01-21 15:00:00,1.10861,0
496,2023-01-21 16:00:00,1.10739,0
497,2023-01-21 17:00:00,1.10713,1
498,2023-01-21 18:00:00,1.10758,1
499,2023-01-21 19:00:00,1.10895,0


In [12]:
# Select Features

X = price[
    [
        "Open",
        "High",
        "Low",
        "Close",
        "Volume",
        "Spread",
        "MA_5",
        "MA_20",
        "RSI",
        "MACD",
        "MACD_Signal",
        "BB_Upper",
        "BB_Middle",
        "BB_Lower",
        "Price_Change",
        "High_Low_Range",
        "Return"
    ]
]

# Target

y = price["Target"]

print("Feature Matrix Shape:", X.shape)

print("Target Shape:", y.shape)

Feature Matrix Shape: (467, 17)
Target Shape: (467,)


In [13]:
# Split the dataset into training and testing sets

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training Features :", X_train.shape)
print("Testing Features  :", X_test.shape)
print("Training Target   :", y_train.shape)
print("Testing Target    :", y_test.shape)

Training Features : (373, 17)
Testing Features  : (94, 17)
Training Target   : (373,)
Testing Target    : (94,)


In [14]:
# Create Random Forest Model

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

# Train the model

model.fit(X_train, y_train)

print("✅ Model Training Completed")

✅ Model Training Completed


In [15]:
# Make predictions on the test data

y_pred = model.predict(X_test)

print("First 10 Predictions:")
print(y_pred[:10])

print("\nFirst 10 Actual Values:")
print(y_test.values[:10])

First 10 Predictions:
[0 0 1 0 1 1 0 1 0 1]

First 10 Actual Values:
[0 1 1 0 0 1 0 0 1 1]


In [16]:
# Calculate model accuracy

accuracy = accuracy_score(y_test, y_pred)

print("Model Accuracy:", round(accuracy * 100, 2), "%")

Model Accuracy: 55.32 %


In [17]:
# Display Classification Report

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.47      0.59      0.52        39
           1       0.64      0.53      0.58        55

    accuracy                           0.55        94
   macro avg       0.56      0.56      0.55        94
weighted avg       0.57      0.55      0.56        94



In [18]:
# Display Confusion Matrix

cm = confusion_matrix(y_test, y_pred)

print(cm)

[[23 16]
 [26 29]]


In [19]:
# Save the trained model

joblib.dump(model, "../models/forex_random_forest.pkl")

print("✅ Model saved successfully!")

✅ Model saved successfully!
